# A especificação do FrodoKEM recorre ao SHAKE256 ou AES para expandir a matriz aleatória 
# A a partir de uma semente com 128 bit (\seedA). Para efeitos deste guião, vamos recorrer directamente às bibliotecas do SageMath ou Python para esse efeito. Deve então codificar a função genMat, que recebe a semente seedA, e retorna uma matriz aleatória de dimensão n×n com elementos amostrados com distribuição uniforme em R


In [1]:
n=640
q=2^(15)
n_=m=8
B=2

In [2]:
def genMat(seedA,q,n):
    R = IntegerModRing(q)
    set_random_seed(seedA)
    A = random_matrix(R,n,n)
    return A

In [3]:
T_CHI640 = [4643, 13363, 20579, 25843, 29227, 31145, 32103, 32525, 32689, 32745, 32762, 32767]
def chiSample(linhas,colunas,q,tabela = T_CHI640):
    R = ZZ
    matriz = matrix(R,linhas,colunas)
    for l in range(linhas):
        for c in range(colunas):
            x = randint(0, 2^15-1)
            i = 0
            while i < len(tabela):
                if tabela[i] > x:
                    break
                i+=1

            bit_sinal = randint(0,1)
            if bit_sinal:
                s = -1
            else:
                s = 1;
            matriz[l, c] = s * i
    return matriz

In [4]:
def encode (bits, n, enq, B = 2): 
    assert len(bits) == n * n * B

    R = Integers(q)
    scale = q // (2^B)

    M = Matrix(R, n, n)

    idx = 0
    for i in range(n):
        for j in range(n):
            # pega B bits
            val = 0
            for b in range(B):
                val = (val << 1) | bits[idx]
                idx += 1

            M[i, j] = R(val * scale)

    return M

In [5]:
def decode (M, q , B = 2):
    n = M.nrows()
    bits = []

    for i in range(n):
        for j in range(n):
            x = Integer(M[i, j])

            val = round(x * (2^B) / q)

            val = val % (2^B)

            for b in reversed(range(B)):
                bits.append((val >> b) & 1)

    return bits

In [6]:
def key_Gen(q,table,n,n_,):
    R = IntegerModRing(q)
    seedA= randint(0,q-1)
    A=genMat(seedA,q,n)
    S = chiSample(n, n_, table)
    E = chiSample(n, n_, table)
    B_pk=A*S+E
    pk=(seedA,B_pk)
    sk=S
    return (sk,pk)

In [7]:
def enc(pk,m,table,n,n_,q):
    R = IntegerModRing(q)
    seedA,B_pk=pk
    A=genMat(seedA,q,n)
    S_=chiSample(n_,n,table)
    E_=chiSample(n_,n,table)
    E__=chiSample(n_,n_,table)
    c1=S_*A+E_
    V_=S_*B_pk+E__
    M = encode(m, n_, q)
    c2=V_+M
    return(c1,c2)

In [8]:
def dec(sk,C,pk,q,n):
    c1,c2=C
    S=sk
    seedA,B_pk=pk
    A=genMat(seedA,q,n)
    V= c1*S
    M=c2-V
    m_=decode(M,q)
    return m_

In [9]:
m_bits = [randint(0, 1) for _ in range(n_ * n_ * B)]

matriz_inicial=encode(m_bits, n_, q, B)
show(matriz_inicial)
sk, pk = key_Gen(q, T_CHI640, n, n_)
C = enc(pk, m_bits, T_CHI640, n, n_, q)
m_ = dec(sk, C, pk, q, n)


c1, c2 = C
V = c1 * sk
matriz_final = c2 - V
show(matriz_final)





[ 8192  8192     0 24576  8192 24576 16384  8192]
[16384 24576 16384     0     0  8192     0 16384]
[24576 24576 24576  8192 24576 16384 16384     0]
[ 8192  8192  8192 16384 24576 16384     0     0]
[ 8192 16384 16384 24576 24576  8192  8192  8192]
[16384     0  8192     0     0 24576     0 24576]
[ 8192 24576 16384     0     0 16384  8192  8192]
[ 8192  8192  8192     0  8192  8192     0 24576]

[ 8583  8013   444 24622  8038 24996 16039  8384]
[16405 25129 16536 32428   358  8848   190 16135]
[24912 24403 24682  8102 24958 16253 16210 32739]
[ 8145  8245  7975 16170 24035 16584 32582 32606]
[ 8660 16410 15984 24614 24179  7893  8281  7958]
[16394     2  8479   161   203 24137 32475 24935]
[ 7995 25011 16407    27   155 16734  7918  8086]
[ 7505  8065  8185 32335  7770  8074   162 24618]

In [10]:
m_bits

[0,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 1,
 1]

In [11]:
m_

[0,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 1,
 1]

In [12]:
m_bits==m_

True